In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

import requests
from langchain_community.tools import DuckDuckGoSearchRun
import os
from dotenv import load_dotenv

load_dotenv()


In [ ]:
search_tool = DuckDuckGoSearchRun()

# result = search_tool.invoke("capital of india")

# result

In [ ]:
API_KEY = os.getenv("WEATHERSTACK_API_KEY")

@tool
def get_weather(city : str) -> str:
    '''This function give current weather of given city'''
    url = f'http://api.weatherstack.com/current'
    
    params = {
        "access_key": API_KEY,
        "query": city
    }
    response = requests.get(url,params=params)
    
    return response.json()

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash"
)

In [ ]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [ ]:
from langchain_core.prompts import PromptTemplate

#creating prompt template due to unable to use prompt from hub

template = """Answer the following question as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: {input}
Thought: You should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

In [ ]:
#creating agent -> it handle all logical or reasoning 

agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather],
    prompt=prompt
)

In [ ]:
#creating agent executor -> it maintain loop or work flow until final answer

agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather],
    verbose=True,
    handle_parsing_errors=True
)

In [ ]:
import os

print("API Key exists:", bool(os.getenv("OPENROUTER_API_KEY")))

In [ ]:

query = input("How can I assist You ?")

result = agent_executor.invoke({"input" : query})

# print(result)

print("AI response : ", result['output'])
    